# 🧠 Extended Thinking & Reasoning with Claude Sonnet 4.6

Welcome to **Meridian Capital**, a mid-size investment firm where the CFO is evaluating deals that can move millions of dollars.

She doesn't just want an answer like *"yes"* or *"no."* She wants to see the reasoning — the way an experienced analyst would sketch assumptions on a whiteboard, weigh trade-offs, and pressure-test downside scenarios before making a call.

In this notebook, you'll build that kind of AI analyst with **Claude Sonnet 4.6** on **Microsoft Foundry**.

## Use case
Meridian Capital needs an AI analyst that can evaluate complex investment opportunities by balancing:
- company fundamentals
- growth vs. burn
- market conditions
- strategic fit
- downside risk
- uncertainty and alternative scenarios

## Learning objectives
By the end, you'll know how to:
1. understand what **extended thinking** is
2. enable it with the Foundry-backed `agent-framework` client
3. inspect the model's **thinking trace** separately from its final answer
4. tune **budget tokens** for shallower or deeper reasoning
5. decide when extended thinking is worth the extra cost and latency


In [4]:
import os
from html import escape

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

from agent_framework import Agent
from agent_framework.anthropic import AnthropicChatOptions
from agent_framework.foundry import AnthropicFoundryClient

load_dotenv(dotenv_path="../.env")

chat_client = AnthropicFoundryClient(
    model=os.environ["FOUNDRY_MODEL_DEPLOYMENT"],
    api_key=os.environ["FOUNDRY_API_KEY"],
    base_url=os.environ["FOUNDRY_ENDPOINT"],
)

base_instructions = """
You are Meridian Capital's senior financial risk analyst.
Give practical, CFO-ready analysis.
Be explicit about assumptions, upside, downside, and uncertainty.
Use crisp business language with light quantitative reasoning.
""".strip()

basic_agent = Agent(
    client=chat_client,
    name="analyst",
    instructions=base_instructions,
)

basic_session = basic_agent.create_session()
quick_test = await basic_agent.run(
    "In one sentence, what should a strong investment memo always include?",
    session=basic_session,
)

print(quick_test.text)


A strong investment memo should always include a clear thesis, key assumptions, expected returns with probability-weighted scenarios, principal risks with mitigants, and a decisive recommendation with defined exit criteria.


## 💭 Standard Responses — The Black Box Problem

When you ask Claude a question normally, you get a polished answer — like receiving a consultant's final report.

It's clean. It's professional. It's useful.

But it's also a **black box**.

You don't see how the model got there.
For low-stakes tasks, that's fine. For a financial decision worth millions, it isn't.

If Meridian is wiring **$5M** into a startup, the CFO needs more than a recommendation. She needs to **see the work**: what risks were weighed, what assumptions were made, and where uncertainty still exists.


In [5]:
financial_question = (
    "Should Meridian Capital invest $5M in a Series B SaaS startup "
    "that has $2M ARR, 40% YoY growth, but negative EBITDA of -$1.5M? "
    "Give a recommendation with key risks and decision criteria."
)

standard_session = basic_agent.create_session()
standard_response = await basic_agent.run(financial_question, session=standard_session)

print(standard_response.text)


# Meridian Capital: Series B SaaS Investment Analysis
## $5M Proposed Investment | Target: Early-Stage SaaS

---

## QUICK VERDICT
**Conditional Pass — Requires Deeper Diligence Before Committing**

The headline metrics are *below* typical Series B thresholds. This isn't a hard no, but the burden of proof sits firmly with the target company.

---

## FINANCIAL SNAPSHOT

| Metric | Target | Series B Benchmark | Gap |
|---|---|---|---|
| ARR | $2M | $5–15M | ⚠️ Undersized |
| YoY Growth | 40% | 80–120%+ | ⚠️ Soft |
| EBITDA Margin | -75% | -20% to -40% | 🔴 Concerning |
| Burn Multiple* | ~0.75x | <1.0x acceptable | ✅ Passable |

*Burn Multiple = Net Burn / Net New ARR. Estimated assuming ~$800K net new ARR and ~$600K quarterly burn.*

---

## THE CORE TENSION

> **The company is burning cash faster than it's building ARR density.**
> At $2M ARR with -$1.5M EBITDA, they're spending roughly **$0.75 to acquire every $1 of recurring revenue.** That's not fatal — but it's tight.

The $5M chec

## 🔓 Extended Thinking — Showing the Work

Extended thinking is like asking that consultant to **think out loud** in front of you.

Instead of disappearing into their office and returning with a polished report, they sit across the table, pull out their notes, and walk through every consideration:

- *"First, let me look at the revenue trajectory..."*
- *"Now let me factor in the burn rate..."*
- *"The market context changes how I should value this..."*
- *"There is upside here, but only if these assumptions hold..."*

You see the raw reasoning, the trade-offs, and even the uncertainty.

In the Anthropic API, you enable this with the `thinking` option:
- `thinking={"type": "enabled", "budget_tokens": ...}` turns extended thinking on
- `budget_tokens` controls how much room Claude has to reason
- `max_tokens` must be **greater** than `budget_tokens`
- when thinking is enabled, **do not set `temperature`**


In [6]:
thinking_options = AnthropicChatOptions(
    thinking={"type": "enabled", "budget_tokens": 5000},
    max_tokens=16000,
)

thinking_agent = Agent(
    client=chat_client,
    name="analyst",
    instructions=base_instructions,
    default_options=thinking_options,
)

thinking_session = thinking_agent.create_session()
thinking_response = await thinking_agent.run(financial_question, session=thinking_session)

thinking_blocks = []
answer_blocks = []

for msg in thinking_response.items:
    for content in msg.contents:
        content_type = str(getattr(content.type, "value", getattr(content.type, "name", ""))).lower()
        if "thinking" in content_type:
            thinking_blocks.append(content.text or str(content.protected_data))
        elif content_type == "text":
            answer_blocks.append(content.text)

print("=== THINKING TRACE ===\n")
print("\n\n".join(thinking_blocks) if thinking_blocks else "[No thinking trace returned]")

print("\n\n=== FINAL ANSWER ===\n")
print("\n\n".join(answer_blocks) if answer_blocks else thinking_response.text)


InternalServerError: Error code: 529 - {'type': 'error', 'error': {'type': 'overloaded_error', 'message': 'Overloaded'}, 'request_id': 'req_011CayE9Nw5V1b8wvParN2At'}

## 🔍 Analyzing the Thinking Trace

When you inspect a thinking trace, look for four things:

1. **Structure of reasoning** — does the model break the problem into sensible steps?
2. **Factors considered** — did it account for growth, burn, valuation logic, market risk, and execution risk?
3. **Assumptions made** — what is it implicitly assuming about future performance?
4. **Uncertainty acknowledged** — does it recognize what it doesn't know?

Think of the thinking trace as an **audit trail**.

In regulated environments like finance, being able to explain **why** the AI recommended something is not just a nice-to-have. It's often part of governance, oversight, and trust.


In [ ]:
def extract_reasoning(response):
    thinking_parts = []
    answer_parts = []

    for msg in response.items:
        for content in msg.contents:
            content_type = str(getattr(content.type, "value", getattr(content.type, "name", ""))).lower()
            if "thinking" in content_type:
                thinking_parts.append(content.text or str(content.protected_data))
            elif content_type == "text":
                answer_parts.append(content.text)

    thinking_text = "\n\n".join(part for part in thinking_parts if part)
    answer_text = "\n\n".join(part for part in answer_parts if part) or response.text
    return thinking_text, answer_text


def preview(text, max_chars=2500):
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n\n...[truncated for display]..."


def display_reasoning(response, title="Reasoning output"):
    thinking_text, answer_text = extract_reasoning(response)

    display(Markdown(f"### {title}"))
    display(Markdown("**Thinking trace**"))
    print(preview(thinking_text) if thinking_text else "[No thinking trace returned]")

    display(Markdown("**Final answer**"))
    print(answer_text)


formatted_session = thinking_agent.create_session()
formatted_response = await thinking_agent.run(financial_question, session=formatted_session)

display_reasoning(formatted_response, title="Series B SaaS investment analysis")


## ⏱️ Budget Tokens — Allocating Thinking Time

`budget_tokens` is like giving your consultant a time allocation.

- A **1,024-token** budget is like saying: *"Give me a quick take in 5 minutes."*
- A **10,000-token** budget is like saying: *"Take the afternoon. I want a thorough review."*

More budget usually means deeper analysis, but it also means:
- higher cost
- longer latency
- more detailed reasoning to review

For Anthropic thinking mode, the minimum thinking budget is typically **1,024 tokens**.


In [ ]:
small_budget_agent = Agent(
    client=chat_client,
    name="analyst-small-budget",
    instructions=base_instructions,
    default_options=AnthropicChatOptions(
        thinking={"type": "enabled", "budget_tokens": 1024},
        max_tokens=4096,
    ),
)

large_budget_agent = Agent(
    client=chat_client,
    name="analyst-large-budget",
    instructions=base_instructions,
    default_options=AnthropicChatOptions(
        thinking={"type": "enabled", "budget_tokens": 10000},
        max_tokens=18000,
    ),
)

small_budget_response = await small_budget_agent.run(
    financial_question,
    session=small_budget_agent.create_session(),
)
large_budget_response = await large_budget_agent.run(
    financial_question,
    session=large_budget_agent.create_session(),
)

small_thinking, small_answer = extract_reasoning(small_budget_response)
large_thinking, large_answer = extract_reasoning(large_budget_response)

comparison_html = f"""
<table style='width:100%; border-collapse:collapse;'>
    <thead>
        <tr>
            <th style='text-align:left; border-bottom:1px solid #ccc; padding:8px;'>Quick take (1,024 tokens)</th>
            <th style='text-align:left; border-bottom:1px solid #ccc; padding:8px;'>Deep dive (10,000 tokens)</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style='vertical-align:top; padding:8px; width:50%;'><pre style='white-space:pre-wrap;'>{escape(preview(small_thinking, 3500))}</pre></td>
            <td style='vertical-align:top; padding:8px; width:50%;'><pre style='white-space:pre-wrap;'>{escape(preview(large_thinking, 3500))}</pre></td>
        </tr>
    </tbody>
</table>
"""

display(HTML(comparison_html))

display(Markdown("### Final answers"))
print("=== 1,024-token answer ===")
print(small_answer)
print("\n=== 10,000-token answer ===")
print(large_answer)


## 📊 Complex Scenario — Full Risk Assessment

Now let's move from a single startup check to something closer to an actual executive decision.

**Scenario:** Meridian Capital is considering acquiring a smaller competitor, **Northstar Analytics**.

This is exactly the kind of messy, multi-factor problem where extended thinking shines.

### What Meridian knows

**Financial profile**
- Meridian Capital revenue: **$120M** with **18% EBITDA margin**
- Northstar Analytics revenue: **$18M ARR** with **22% YoY growth**
- Northstar EBITDA margin: **8%**
- Asking price: **$52M**
- Expected integration costs: **$6M** over 18 months
- Projected annual synergies by year 3: **$9M**

**Market position**
- Meridian is #3 in its niche
- Northstar is #5
- Combined entity would reach about **18% market share**
- Two larger incumbents still control more than 50% of the market

**Risk factors**
- Northstar has **28% customer concentration** in its top 3 clients
- A key EU product line may require **new regulatory compliance investment** next year
- Northstar's founder wants to exit after the acquisition
- The CTO is strong but may leave without a retention package
- Product overlap is moderate, which creates both cross-sell upside and integration complexity

A human CFO would not solve this with one ratio. They would build a mosaic from finance, strategy, people risk, regulation, and execution. That's what we'll ask Claude to do next.


In [ ]:
acquisition_prompt = """
Meridian Capital is evaluating whether to acquire Northstar Analytics.

Please act like a senior M&A and risk analyst for the CFO.

Facts:
- Meridian revenue: $120M, EBITDA margin 18%
- Northstar revenue: $18M ARR, growth 22% YoY, EBITDA margin 8%
- Asking price: $52M
- Integration cost: $6M over 18 months
- Expected annual synergies by year 3: $9M
- Combined market share would be about 18%
- Two larger incumbents still control more than 50% of the market
- Northstar top 3 customers represent 28% of revenue
- EU regulatory compliance costs may rise next year for a key product line
- Founder plans to exit after acquisition
- CTO is strong but is a retention risk without incentives
- Product overlap is moderate: some cross-sell upside, some integration complexity

Analyze this decision from multiple angles:
1. strategic fit
2. financial attractiveness
3. execution and integration risk
4. regulatory and customer concentration risk
5. best-case, base-case, and downside scenario

Then give a final recommendation:
- Acquire now
- Negotiate different terms
- Wait and monitor
- Pass

Be explicit about your assumptions and the biggest swing factors.
""".strip()

complex_agent = Agent(
    client=chat_client,
    name="m_and_a_analyst",
    instructions=base_instructions,
    default_options=AnthropicChatOptions(
        thinking={"type": "enabled", "budget_tokens": 12000},
        max_tokens=20000,
    ),
)

complex_response = await complex_agent.run(
    acquisition_prompt,
    session=complex_agent.create_session(),
)

display_reasoning(complex_response, title="Acquisition risk assessment for Meridian Capital")


## ⚖️ When to Use Extended Thinking

Extended thinking is powerful, but it is not the default answer to every problem.

### ✅ Use it when
- the task requires **complex multi-step reasoning**
- the decision is **high stakes**
- you need an **audit trail** for trust, governance, or review
- the problem is ambiguous and trade-off heavy
- the model needs to compare scenarios, assumptions, and second-order effects

### ❌ Skip it when
- you're doing a simple factual lookup
- you need quick, lightweight text generation
- you're in a cost-sensitive, high-volume workflow
- latency matters more than transparency

### Trade-offs to remember
- extended thinking increases **cost**
- extended thinking increases **latency**
- `temperature` cannot be used when thinking is enabled
- `max_tokens` must be greater than `budget_tokens`

A good mental model: use extended thinking when you would normally say, *"Show me your spreadsheet, not just your conclusion."*


## 🎯 Try It Yourself

Here are a few strong follow-up exercises:

1. **Legal contract review**  
   Ask Claude to reason through indemnification clauses, termination risk, and negotiation leverage.

2. **Technical architecture review**  
   Give it a migration plan and ask it to evaluate reliability, scalability, security, and implementation risk.

3. **Budget tuning experiment**  
   Run the same prompt with `1024`, `5000`, and `10000` thinking budgets, then compare depth, latency, and usefulness.

## Key takeaways
- **Standard mode** gives you the polished memo.
- **Extended thinking** lets you inspect the analyst's scratchpad.
- **Budget tokens** control how much room Claude has to reason.
- The best use cases are **complex, ambiguous, high-stakes decisions**.
- In Foundry, this becomes a practical way to build AI systems that are not just smart, but also easier to trust.
